# Measurement Error Mitigation — Readout Error Calibration & Correction

Measurement errors (SPAM — State Preparation and Measurement) create a noise floor on physical devices. Readout errors occur when a qubit in state $|0\rangle$ is measured as $|1\rangle$ or vice-versa.

In this notebook, we calibrate measurement noise by constructing a readout confusion matrix and apply a Moore-Penrose pseudo-inverse mitigation method to reconstruct noiseless probabilities.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_mitigation import (
    grover_unitary, qft_unitary,
    noisy_simulator, ideal_simulator,
    build_calibration_matrix, apply_measurement_mitigation,
    run_circuit, total_variation_distance, hellinger_fidelity,
    plot_calibration_matrix, plot_comparison_bar
)
%matplotlib inline


## 1. Building the Calibration Matrix

We prepare all computational basis states ($|00\rangle, |01\rangle, \dots$) and measure them on our noisy simulator to build the measurement calibration matrices for 2 and 3 qubits.


In [ ]:
sim_noisy = noisy_simulator()

# Build matrices
cal_matrix_2q = build_calibration_matrix(sim_noisy, 2)
cal_matrix_3q = build_calibration_matrix(sim_noisy, 3)

print(f"2-Qubit Diagonal Mean: {np.diag(cal_matrix_2q).mean():.4f}")
print(f"3-Qubit Diagonal Mean: {np.diag(cal_matrix_3q).mean():.4f}")

# Plot 2-qubit matrix
plot_calibration_matrix(cal_matrix_2q, title="2-Qubit Calibration Matrix")
plt.show()


## 2. Correcting Grover's Search

We execute Grover's Search under simulated readout noise and correct the measurement probabilities.


In [ ]:
qc_grover = grover_unitary()
qc_grover.measure_all()

# Run noisy simulation
counts_unmit, probs_unmit = run_circuit(qc_grover, sim_noisy)

# Mitigate
probs_mit = apply_measurement_mitigation(counts_unmit, cal_matrix_2q)

print("Grover's Search Readout Mitigation Summary:")
print(f"  Ideal P(|11⟩)      = 1.0000")
print(f"  Unmitigated P(|11⟩) = {probs_unmit[3]:.4f} (error = {abs(1.0 - probs_unmit[3]):.4f})")
print(f"  Mitigated P(|11⟩)   = {probs_mit[3]:.4f} (error = {abs(1.0 - probs_mit[3]):.4f})")
print(f"  Readout error reduction: {((1.0 - probs_unmit[3]) - (1.0 - probs_mit[3])) / (1.0 - probs_unmit[3]) * 100:.1f}%")

# Plot bar comparison
plot_comparison_bar(['Unmitigated', 'Mitigated', 'Ideal'], [probs_unmit[3], probs_mit[3], 1.0], title="Readout Mitigation on Grover's Search")
plt.show()


## 3. Correcting the 3-Qubit QFT

We measure TVD and Hellinger Fidelity improvement on a 3-qubit QFT circuit under readout correction.


In [ ]:
qc_qft = qft_unitary(3)
qc_qft.measure_all()

# Run ideal noiseless simulator
_, ideal_probs = run_circuit(qc_qft, ideal_simulator())

# Run noisy simulator
counts_unmit_qft, probs_unmit_qft = run_circuit(qc_qft, sim_noisy)

# Mitigate
probs_mit_qft = apply_measurement_mitigation(counts_unmit_qft, cal_matrix_3q)

tvd_unmit = total_variation_distance(probs_unmit_qft, ideal_probs)
tvd_mit = total_variation_distance(probs_mit_qft, ideal_probs)
fid_unmit = hellinger_fidelity(probs_unmit_qft, ideal_probs)
fid_mit = hellinger_fidelity(probs_mit_qft, ideal_probs)

print("3-Qubit QFT Readout Mitigation Summary (Ideal = Uniform Distribution):\n")
print(f"  Unmitigated TVD        = {tvd_unmit:.4f}  |  Hellinger Fidelity = {fid_unmit:.6f}")
print(f"  Mitigated TVD          = {tvd_mit:.4f}  |  Hellinger Fidelity = {fid_mit:.6f}")
print("-" * 70)
print(f"  TVD Error Reduction   = {(tvd_unmit - tvd_mit) / tvd_unmit * 100:.1f}%")
print(f"  Fidelity Gap Reduction = {(fid_mit - fid_unmit) / (1.0 - fid_unmit) * 100:.1f}%")
